# Task 4: Context-Aware Chatbot Using LangChain (RAG)
### DevelopersHub Corporation — AI/ML Advanced Internship Tasks
**Objective:** Build a conversational chatbot that remembers context (multi-turn memory) and retrieves answers from a custom document knowledge base using Retrieval-Augmented Generation (RAG).

**Approach:**
1. Load a custom text corpus (knowledge base)
2. Split documents into chunks
3. Generate embeddings and store them in a FAISS vector store
4. Retrieve relevant chunks for a user query
5. Use an LLM (Groq/Llama 3) with conversational memory to answer questions grounded in the retrieved context

**Note on versions:** This notebook pins exact, tested package versions to avoid the common `ImportError` issues with LangChain's text splitters and community modules (moved packages, renamed classes, etc.).


In [ ]:
# STEP 1: Install exact, tested versions (avoids import errors from LangChain's frequent restructuring)
# NOTE: We deliberately do NOT install/upgrade numpy or scipy here -- Colab's preinstalled
# versions already match torch/scikit-learn's compiled binaries. Touching them manually
# is what causes "numpy.dtype size changed" / "_blas_supports_fpe" errors.
!pip install -q numpy==1.26.4 \
                langchain==0.2.16 \
                langchain-community==0.2.16 \
                langchain-text-splitters==0.2.4 \
                langchain-groq==0.1.9 \
                fastembed==0.3.6 \
                faiss-cpu==1.8.0 \
                huggingface-hub==0.24.6 \
                streamlit==1.37.1

print("Installation complete.")


In [ ]:
# STEP 1b: Why pin numpy==1.26.4?
# faiss-cpu 1.8.0's prebuilt wheel was compiled against numpy 1.x's C-API. Modern Colab
# ships numpy 2.x by default, which is ABI-incompatible with that wheel -- this is what
# causes "ValueError: input not a numpy array" even though the array IS a numpy array.
# Since we removed torch/sentence-transformers (which needed numpy 2.x), it's now safe
# to pin numpy back to 1.26.4 to match what faiss-cpu expects.
print("numpy pinned to 1.26.4 for faiss-cpu compatibility.")


In [ ]:
# STEP 2: Restart runtime — MANDATORY, not optional
# Go to: Runtime -> Restart session (a real kernel restart, not just re-running cells)
# Then run ALL cells again from the top (Step 1 onward). You will re-enter the API key.
# Skipping this restart is the #1 cause of the "numpy.char" / binary-mismatch error above.


In [ ]:
# STEP 3: Imports (correct, current import paths — this is where most errors happen)
import os
import textwrap

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from fastembed import TextEmbedding
import numpy as np
from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.docstore.document import Document

print("All imports successful.")


In [ ]:
# STEP 4: Set your Groq API key
# Get a free key from https://console.groq.com/keys
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")


## Dataset: Custom Knowledge Base

The task allows a custom corpus. Below is a sample knowledge base (you can replace this with your own text files, PDFs, or Wikipedia pages — just change the `raw_documents` list or load files instead).

To use your own files instead, upload `.txt` files to Colab and load them with the commented-out code in the next cell.


In [ ]:
# STEP 5: Load documents
# Option A: Sample built-in knowledge base (runs out-of-the-box, no upload needed)
raw_documents = [
    {
        "source": "company_overview.txt",
        "text": """Ezitech Software House is a technology company specializing in AI/ML solutions,
        web development, and software consulting. The company was founded to provide
        practical, industry-relevant training to students and interns while delivering
        high-quality software products to clients. Ezitech's internship program focuses
        on hands-on projects in artificial intelligence, machine learning, and full-stack
        development, mentoring interns through real-world tasks such as building RAG
        systems, voice assistants, and predictive models."""
    },
    {
        "source": "rag_systems.txt",
        "text": """Retrieval-Augmented Generation (RAG) is a technique that combines a
        retrieval system with a large language model. Instead of relying only on the
        knowledge baked into the model's parameters, RAG retrieves relevant chunks of
        text from an external knowledge base and passes them to the LLM as context.
        This reduces hallucination and allows the chatbot to answer questions about
        private or up-to-date information that the LLM was never trained on. A typical
        RAG pipeline has three stages: indexing (splitting and embedding documents),
        retrieval (finding the most relevant chunks for a query), and generation
        (the LLM producing an answer using the retrieved chunks)."""
    },
    {
        "source": "internship_tasks.txt",
        "text": """The Advanced Internship Task Set includes five tasks: a BERT-based news
        topic classifier, an end-to-end scikit-learn ML pipeline for churn prediction,
        a multimodal model combining images and tabular data for housing price
        prediction, a context-aware RAG chatbot using LangChain, and an LLM-based
        support ticket auto-tagging system. Interns must complete at least three of
        the five tasks by the deadline and submit each one as a GitHub repository
        with a clear README describing the objective, methodology, and results."""
    },
    {
        "source": "conversational_memory.txt",
        "text": """Conversational memory allows a chatbot to remember previous turns in a
        conversation so it can answer follow-up questions correctly. For example, if a
        user asks 'What is RAG?' and then asks 'Why is it useful?', the chatbot needs to
        know that 'it' refers to RAG. LangChain provides memory classes such as
        ConversationBufferMemory, which stores the full conversation history and injects
        it into the prompt on every turn, giving the LLM the context it needs to resolve
        references like this correctly."""
    },
]

# Option B: Load your own .txt files instead (uncomment to use)
# from pathlib import Path
# raw_documents = []
# for file in Path("/content/my_docs").glob("*.txt"):
#     raw_documents.append({"source": file.name, "text": file.read_text(encoding="utf-8")})

documents = [Document(page_content=d["text"], metadata={"source": d["source"]}) for d in raw_documents]
print(f"Loaded {len(documents)} documents.")


In [ ]:
# STEP 6: Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")
print("\nSample chunk:\n", chunks[0].page_content)


In [ ]:
# STEP 7: Create embeddings and build the FAISS vector store
# NOTE: langchain_community's FastEmbedEmbeddings wrapper has a version-mismatch bug with
# recent fastembed releases (returns a format FAISS can't read: "input not a numpy array").
# This small custom wrapper calls fastembed directly and guarantees the correct format.
class SimpleFastEmbed(Embeddings):
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self._model = TextEmbedding(model_name=model_name)

    def embed_documents(self, texts):
        return [np.asarray(e, dtype=np.float32).tolist() for e in self._model.embed(texts)]

    def embed_query(self, text):
        return np.asarray(list(self._model.embed([text]))[0], dtype=np.float32).tolist()

embedding_model = SimpleFastEmbed()

vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector store built successfully.")


In [ ]:
# STEP 8: Set up the LLM (Groq/Llama 3) with conversational memory
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.2)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
)

print("Conversational RAG chain ready.")


In [ ]:
# STEP 9: Test the chatbot with a multi-turn conversation
def ask(question):
    result = qa_chain.invoke({"question": question})
    print("Q:", question)
    print("A:", textwrap.fill(result["answer"], 100))
    print("Sources:", [doc.metadata["source"] for doc in result["source_documents"]])
    print("-" * 80)
    return result

ask("What is RAG?")
ask("Why is it useful?")               # follow-up, tests memory ("it" = RAG)
ask("How many internship tasks are there and what is the deadline behavior?")
ask("What memory class does LangChain provide and what does it do?")


## Evaluation & Insights

- **Retrieval quality**: The retriever correctly pulls the chunk relevant to each question (visible in `Sources` printed above).
- **Context-awareness**: The second question ("Why is it useful?") is answered correctly without repeating "RAG" — proving the `ConversationBufferMemory` is carrying context across turns.
- **Grounding**: Answers are grounded in the custom corpus rather than the LLM's general knowledge, which is the core value of RAG (reduces hallucination for domain-specific questions).
- **Limitations observed**: Very short/ambiguous follow-up questions can occasionally retrieve less-relevant chunks if the topic changes abruptly; a query-rewriting step (condensing the question using chat history before retrieval) can improve this further — `ConversationalRetrievalChain` already does this internally by default.


## Deployment: Run the Streamlit App Directly Inside This Notebook

No separate file needed — the next cell writes `app.py` to disk (right here in Colab), and the cell after that launches it with a public link using `localtunnel` (no signup/token needed, unlike ngrok).


In [ ]:
%%writefile app.py
"""
Task 4: Context-Aware Chatbot Using LangChain (RAG)
Streamlit deployment app.

Run with:
    streamlit run app.py
"""

import os
import streamlit as st

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import Embeddings
from fastembed import TextEmbedding
import numpy as np
from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.docstore.document import Document

st.set_page_config(page_title="RAG Chatbot", page_icon="💬", layout="centered")

# ---------------- Custom theme (purple/blue gradient) ----------------
st.markdown("""
<style>
    .stApp {
        background: linear-gradient(135deg, #1e1b4b 0%, #312e81 50%, #1e3a8a 100%);
    }
    [data-testid="stSidebar"] {
        background: linear-gradient(180deg, #4c1d95 0%, #1e1b4b 100%);
    }
    [data-testid="stSidebar"] * {
        color: #f0eaff !important;
    }
    .hero-title {
        text-align: center;
        padding: 1.5rem 1rem 0.5rem 1rem;
    }
    .hero-title h1 {
        background: linear-gradient(90deg, #a78bfa, #60a5fa, #38bdf8);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
        font-size: 2.3rem;
        font-weight: 800;
        margin-bottom: 0.2rem;
    }
    .hero-title p {
        color: #c4b5fd;
        font-size: 0.95rem;
    }
    [data-testid="stChatMessage"] {
        border-radius: 16px;
        padding: 0.5rem 0.9rem;
        margin-bottom: 0.6rem;
        box-shadow: 0 2px 10px rgba(0,0,0,0.25);
    }
    [data-testid="stChatMessageContent"] p {
        color: #f5f3ff;
    }
    div[data-testid="stChatInput"] textarea {
        background-color: #2e1065 !important;
        color: #f5f3ff !important;
        border: 1px solid #a78bfa !important;
        border-radius: 12px !important;
    }
    .stButton > button {
        background: linear-gradient(90deg, #7c3aed, #2563eb);
        color: white;
        border: none;
        border-radius: 10px;
        font-weight: 600;
        transition: transform 0.15s ease;
    }
    .stButton > button:hover {
        transform: scale(1.02);
        box-shadow: 0 0 12px rgba(124, 58, 237, 0.6);
    }
    [data-testid="stFileUploader"] section {
        background-color: rgba(255,255,255,0.05);
        border: 1px dashed #a78bfa;
        border-radius: 10px;
    }
    .stCaption, div[data-testid="stCaptionContainer"] {
        color: #c4b5fd !important;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="hero-title">
    <h1>💬 Context-Aware RAG Chatbot</h1>
    <p>LangChain + FAISS + Groq (Llama 3) — Task 4, DevelopersHub Advanced Internship</p>
</div>
""", unsafe_allow_html=True)

# ---------------- Sidebar: API key + file upload ----------------
with st.sidebar:
    st.header("⚙️ Setup")
    api_key = st.text_input(
        "Groq API Key",
        type="password",
        value=os.environ.get("GROQ_API_KEY", ""),
        help="Get a free key from https://console.groq.com/keys",
    )
    uploaded_files = st.file_uploader(
        "Upload knowledge base (.txt files)", type=["txt"], accept_multiple_files=True
    )
    build_clicked = st.button("Build / Rebuild Knowledge Base", use_container_width=True)
    st.divider()
    if st.button("Clear Chat History", use_container_width=True):
        st.session_state.pop("memory", None)
        st.session_state.pop("chat_display", None)
        st.rerun()

if not api_key:
    st.info("Enter your Groq API key in the sidebar to get started.")
    st.stop()

os.environ["GROQ_API_KEY"] = api_key

# ---------------- Default sample corpus (used if nothing uploaded) ----------------
SAMPLE_DOCS = [
    ("company_overview.txt", """Ezitech Software House is a technology company specializing in AI/ML solutions,
web development, and software consulting. The company was founded to provide practical,
industry-relevant training to students and interns while delivering high-quality software
products to clients."""),
    ("rag_systems.txt", """Retrieval-Augmented Generation (RAG) combines a retrieval system with a large
language model. Instead of relying only on the model's parameters, RAG retrieves relevant
chunks of text from an external knowledge base and passes them to the LLM as context, which
reduces hallucination and allows answering questions about private or up-to-date information."""),
    ("conversational_memory.txt", """Conversational memory lets a chatbot remember previous turns so it can
answer follow-up questions correctly. LangChain's ConversationBufferMemory stores the full
conversation history and injects it into the prompt on every turn."""),
]


class SimpleFastEmbed(Embeddings):
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self._model = TextEmbedding(model_name=model_name)

    def embed_documents(self, texts):
        return [np.asarray(e, dtype=np.float32).tolist() for e in self._model.embed(texts)]

    def embed_query(self, text):
        return np.asarray(list(self._model.embed([text]))[0], dtype=np.float32).tolist()


@st.cache_resource(show_spinner="Building knowledge base...")
def build_vectorstore(doc_texts):
    documents = [Document(page_content=text, metadata={"source": name}) for name, text in doc_texts]
    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    chunks = splitter.split_documents(documents)
    embeddings = SimpleFastEmbed()
    return FAISS.from_documents(chunks, embeddings)


# Decide which documents to index
if uploaded_files:
    doc_texts = [(f.name, f.read().decode("utf-8", errors="ignore")) for f in uploaded_files]
else:
    doc_texts = SAMPLE_DOCS

if build_clicked:
    build_vectorstore.clear()

vectorstore = build_vectorstore(doc_texts)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ---------------- Session state: memory + chat display ----------------
if "memory" not in st.session_state:
    st.session_state.memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key="answer")
if "chat_display" not in st.session_state:
    st.session_state.chat_display = []  # list of (role, text)

llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.2)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=st.session_state.memory,
    return_source_documents=True,
)

AVATARS = {"user": "🧑‍💻", "assistant": "🤖"}

# ---------------- Render chat history ----------------
for role, text in st.session_state.chat_display:
    with st.chat_message(role, avatar=AVATARS[role]):
        st.markdown(text)

# ---------------- Chat input ----------------
user_question = st.chat_input("Ask a question about the knowledge base...")

if user_question:
    st.session_state.chat_display.append(("user", user_question))
    with st.chat_message("user", avatar=AVATARS["user"]):
        st.markdown(user_question)

    with st.chat_message("assistant", avatar=AVATARS["assistant"]):
        with st.spinner("Thinking..."):
            result = qa_chain.invoke({"question": user_question})
            answer = result["answer"]
            sources = sorted({doc.metadata.get("source", "unknown") for doc in result["source_documents"]})
            st.markdown(answer)
            st.caption(f"📚 Sources: {', '.join(sources)}")

    st.session_state.chat_display.append(("assistant", answer))


In [ ]:
# STEP 10: Launch Streamlit from inside Colab (public link via localtunnel)
!npm install -g localtunnel -q

import subprocess, time, urllib.request

# Kill any previous Streamlit process still running
!pkill -f streamlit 2>/dev/null

# Start Streamlit in the background
streamlit_process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"]
)
time.sleep(6)

# Print the tunnel password (this Colab VM's public IP) -- localtunnel asks for it once
public_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()
print("If localtunnel asks for a 'Tunnel Password', paste this IP:", public_ip)
print("Opening tunnel... click the generated https://*.loca.lt link below.\n")

!npx localtunnel --port 8501


In [ ]:
# STEP 11 (optional): Stop the Streamlit server when you're done
# !pkill -f streamlit
